# 01 · Build the clinical graph

Loads the **MIMIC-IV Clinical Database Demo v2.2** (ODC-BY 1.0, open access —
no credentialing) into a Neo4j property graph.

Produces **33,980 nodes and 45,226 relationships** over 100 patients and 275
admissions. Runs in roughly eight minutes on a free Colab instance against an
AuraDB Free tier.

**Requires:** a Neo4j AuraDB instance (URI, username, password). Credentials are
entered at runtime with `getpass` and never written to disk.

**Run top to bottom.** Section 4 is not idempotent — `LabEvent` is created with
`CREATE`, not `MERGE`, because lab results have no natural unique key. Running
it twice doubles the lab data. The guard at the top of section 4 refuses to
proceed if any `LabEvent` already exists.

## 1 · Download the demo release

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/graphrag_clinical/'
os.makedirs(BASE, exist_ok=True)
%cd {BASE}

# Open access — no credentials needed for the demo release.
!wget -q -r -N -c -np --show-progress https://physionet.org/files/mimic-iv-demo/2.2/

ROOT = BASE + 'physionet.org/files/mimic-iv-demo/2.2/'
print("done")

## 2 · Inspect the release

Optional. Confirms the download and shows the admission-count distribution that makes longitudinal queries possible.

In [ ]:
import pandas as pd

pat = pd.read_csv(ROOT + 'hosp/patients.csv.gz')
adm = pd.read_csv(ROOT + 'hosp/admissions.csv.gz')
counts = adm.groupby('subject_id').size().sort_values(ascending=False)

print(f"patients      : {len(pat)}")
print(f"admissions    : {len(adm)}")
print(f"2+ admissions : {(counts >= 2).sum()}")
print(f"3+ admissions : {(counts >= 3).sum()}")

for name in ['diagnoses_icd', 'procedures_icd', 'labevents', 'prescriptions']:
    df = pd.read_csv(ROOT + f'hosp/{name}.csv.gz')
    print(f"{name:<16}{len(df):>9,} rows   "
          f"{len(df)/df.subject_id.nunique():>7.0f} per patient")

## 3 · Connect to AuraDB

For an Aura instance the username and database name are both the instance id (the leading label of the connection URI).

In [ ]:
!pip install -q neo4j

from neo4j import GraphDatabase
from getpass import getpass

URI  = input("AuraDB connection URI (neo4j+s://<id>.databases.neo4j.io): ").strip()
USER = input("username (usually the instance id): ").strip()
DB   = USER

driver = GraphDatabase.driver(URI, auth=(USER, getpass("password: ")))
driver.verify_connectivity()


def run(q, rows=None, batch=500):
    """Execute Cypher. With `rows`, UNWIND them in batches."""
    with driver.session(database=DB) as s:
        if rows is None:
            return [r.data() for r in s.run(q)]
        for i in range(0, len(rows), batch):
            s.run(q, rows=rows[i:i+batch])


print("connected ·", run("MATCH (n) RETURN count(n) AS n")[0]["n"], "nodes")

## 4 · Load

**Not idempotent.** The guard below stops the cell if `LabEvent` nodes already
exist, because `CREATE` would duplicate them. To rebuild from scratch:

```cypher
MATCH (n) CALL { WITH n DETACH DELETE n } IN TRANSACTIONS OF 10000 ROWS
```

`clean()` coerces pandas `NaN` to `None` before every load. Without it, NaN
reaches Neo4j as a float NaN rather than a null: `MERGE` rejects it outright on
a relationship property, and `SET` accepts it silently — after which
`WHERE x IS NULL` no longer matches those rows, because NaN compares unequal to
everything including itself. An earlier build of this graph carried 663 such
properties.

In [ ]:
existing = run("MATCH (l:LabEvent) RETURN count(l) AS n")[0]["n"]
assert existing == 0, (
    f"{existing:,} LabEvent nodes already exist. Loading again would double "
    "them — wipe the graph first (see the note above).")


def load(t):
    return pd.read_csv(ROOT + f'hosp/{t}.csv.gz')


def clean(df):
    """pandas NaN -> None, so Neo4j stores nulls rather than float NaN."""
    return df.astype(object).where(df.notna(), None)


pat  = load('patients');       adm  = load('admissions')
dx   = load('diagnoses_icd');  ddx  = load('d_icd_diagnoses')
proc = load('procedures_icd'); dpr  = load('d_icd_procedures')
rx   = load('prescriptions');  lab  = load('labevents')
dlab = load('d_labitems')

for c in [
  "CREATE CONSTRAINT IF NOT EXISTS FOR (p:Patient)    REQUIRE p.subject_id IS UNIQUE",
  "CREATE CONSTRAINT IF NOT EXISTS FOR (a:Admission)  REQUIRE a.hadm_id    IS UNIQUE",
  "CREATE CONSTRAINT IF NOT EXISTS FOR (d:Diagnosis)  REQUIRE d.icd_code   IS UNIQUE",
  "CREATE CONSTRAINT IF NOT EXISTS FOR (p:Procedure)  REQUIRE p.icd_code   IS UNIQUE",
  "CREATE CONSTRAINT IF NOT EXISTS FOR (m:Medication) REQUIRE m.name       IS UNIQUE",
  "CREATE INDEX IF NOT EXISTS FOR (l:LabEvent) ON (l.label)",
]:
    run(c)
print("constraints and index in place")

# ---- patients ------------------------------------------------------------
run("""UNWIND $rows AS r
MERGE (p:Patient {subject_id: r.subject_id})
SET p.gender = r.gender, p.anchor_age = r.anchor_age,
    p.anchor_year = r.anchor_year""", clean(pat).to_dict('records'))
print("patients loaded")

# ---- admissions ----------------------------------------------------------
run("""UNWIND $rows AS r
MATCH (p:Patient {subject_id: r.subject_id})
MERGE (a:Admission {hadm_id: r.hadm_id})
SET a.admission_type = r.admission_type, a.admittime = r.admittime,
    a.dischtime = r.dischtime, a.discharge_location = r.discharge_location
MERGE (p)-[:HAS_ADMISSION]->(a)""", clean(adm).to_dict('records'))
print("admissions loaded")

# ---- diagnoses -----------------------------------------------------------
dx2 = dx.merge(ddx, on=['icd_code', 'icd_version'], how='left')
dx2['long_title'] = dx2.long_title.fillna(dx2.icd_code)
run("""UNWIND $rows AS r
MATCH (a:Admission {hadm_id: r.hadm_id})
MERGE (d:Diagnosis {icd_code: r.icd_code})
SET d.long_title = r.long_title, d.icd_version = r.icd_version
MERGE (a)-[:HAS_DIAGNOSIS {seq_num: r.seq_num}]->(d)""",
    clean(dx2).to_dict('records'))
print("diagnoses loaded")

# ---- procedures ----------------------------------------------------------
pr2 = proc.merge(dpr, on=['icd_code', 'icd_version'], how='left')
pr2['long_title'] = pr2.long_title.fillna(pr2.icd_code)
run("""UNWIND $rows AS r
MATCH (a:Admission {hadm_id: r.hadm_id})
MERGE (p:Procedure {icd_code: r.icd_code})
SET p.long_title = r.long_title
MERGE (a)-[:HAS_PROCEDURE {chartdate: r.chartdate}]->(p)""",
    clean(pr2).to_dict('records'))
print("procedures loaded")

# ---- medications ---------------------------------------------------------
rx2 = rx.dropna(subset=['drug', 'hadm_id']).copy()
rx2['drug'] = rx2.drug.str.lower().str.strip()
run("""UNWIND $rows AS r
MATCH (a:Admission {hadm_id: r.hadm_id})
MERGE (m:Medication {name: r.drug})
MERGE (a)-[h:HAS_MEDICATION]->(m)
SET h.route     = r.route,
    h.dose      = r.dose_val_rx,
    h.unit      = r.dose_unit_rx,
    h.starttime = r.starttime""", clean(rx2).to_dict('records'))
print("medications loaded")

# ---- abnormal labs only --------------------------------------------------
# CREATE, not MERGE: lab results have no natural unique key. This is the cell
# that must not run twice.
lab2 = (lab[lab.flag == 'abnormal']
        .dropna(subset=['hadm_id'])
        .merge(dlab[['itemid', 'label']], on='itemid', how='left'))
run("""UNWIND $rows AS r
MATCH (a:Admission {hadm_id: r.hadm_id})
CREATE (l:LabEvent {label: r.label, value: r.value, valuenum: r.valuenum,
                    valueuom: r.valueuom, flag: r.flag, charttime: r.charttime})
MERGE (a)-[:INCLUDES_LAB]->(l)""", clean(lab2).to_dict('records'))
print("labs loaded")

## 5 · Verify

Expected: 33,980 nodes, 45,226 relationships, and no NaN-valued properties.

In [ ]:
print(run("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS n ORDER BY n DESC"))
print(run("MATCH ()-[r]->() RETURN type(r) AS rel, count(*) AS n ORDER BY n DESC"))
print("nodes:", run("MATCH (n) RETURN count(n) AS n")[0]["n"])
print("rels :", run("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"])

nan = run("""MATCH (n) UNWIND keys(n) AS k
WITH n[k] AS v WHERE toString(v) = 'NaN'
RETURN count(*) AS nan_properties""")[0]["nan_properties"]
print("NaN properties:", nan, "(expected 0)")